Conjunto de Datos 1: daily-total-female-births.csv


# 1. Análisis Exploratorio:

## ProfileReport

In [1]:
import pandas as pd
from ydata_profiling import ProfileReport

# Cargar datos
df = pd.read_csv('./daily-total-female-births.csv')

# Convertir fecha
df['Date'] = pd.to_datetime(df['Date'])

# Generar reporte
profile = ProfileReport(
    df,
    title="Análisis de Nacimientos",
    explorative=True
)

# Guardar reporte
profile.to_file("reporte_births.html")

print("✅ Reporte generado: reporte_births.html")
print(f"📊 Datos analizados: {len(df)} registros desde {df['Date'].min()} hasta {df['Date'].max()}")

c:\Users\villa\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 328.99it/s]

✅ Reporte generado: reporte_births.html
📊 Datos analizados: 365 registros desde 1959-01-01 00:00:00 hasta 1959-12-31 00:00:00


In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Cargar datos
df = pd.read_csv('./daily-total-female-births.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Crear variables temporales adicionales
df['DayOfWeek'] = df.index.dayofweek
df['DayName'] = df.index.day_name()
df['Month'] = df.index.month
df['MonthName'] = df.index.month_name()
df['Quarter'] = df.index.quarter
df['DayOfYear'] = df.index.dayofyear
df['WeekOfYear'] = df.index.isocalendar().week

# 1. ESTADÍSTICAS DESCRIPTIVAS
print("=" * 60)
print("ANÁLISIS EXPLORATORIO DE NACIMIENTOS FEMENINOS DIARIOS - 1959")
print("=" * 60)

print("\n1. ESTADÍSTICAS DESCRIPTIVAS:")
print("-" * 40)
stats_summary = df['Births'].describe()
print(stats_summary)
print(f"\nAsimetría (Skewness): {df['Births'].skew():.3f}")
print(f"Curtosis: {df['Births'].kurtosis():.3f}")
print(f"Coeficiente de variación: {(df['Births'].std() / df['Births'].mean() * 100):.2f}%")

# 2. ANÁLISIS TEMPORAL
print("\n2. ANÁLISIS TEMPORAL:")
print("-" * 40)

# Estadísticas por día de la semana
day_stats = df.groupby('DayName')['Births'].agg(['mean', 'std', 'count'])
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_stats = day_stats.reindex(day_order)
print("\nNacimientos por día de la semana:")
print(day_stats.round(2))

# Estadísticas por mes
month_stats = df.groupby('MonthName')['Births'].agg(['mean', 'std', 'sum'])
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']
month_stats = month_stats.reindex(month_order)
print("\nNacimientos por mes:")
print(month_stats.round(2))

# 3. ANÁLISIS DE OUTLIERS
print("\n3. ANÁLISIS DE OUTLIERS:")
print("-" * 40)
Q1 = df['Births'].quantile(0.25)
Q3 = df['Births'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Births'] < lower_bound) | (df['Births'] > upper_bound)]
print(f"Límites IQR: [{lower_bound:.1f}, {upper_bound:.1f}]")
print(f"Número de outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")
if len(outliers) > 0:
    print("\nFechas con valores atípicos:")
    for idx, row in outliers.iterrows():
        print(f"  {idx.strftime('%Y-%m-%d')} ({idx.strftime('%A')}): {row['Births']} nacimientos")


ANÁLISIS EXPLORATORIO DE NACIMIENTOS FEMENINOS DIARIOS - 1959

1. ESTADÍSTICAS DESCRIPTIVAS:
----------------------------------------
count    365.000000
mean      41.980822
std        7.348257
min       23.000000
25%       37.000000
50%       42.000000
75%       46.000000
max       73.000000
Name: Births, dtype: float64

Asimetría (Skewness): 0.447
Curtosis: 0.778
Coeficiente de variación: 17.50%

2. ANÁLISIS TEMPORAL:
----------------------------------------

Nacimientos por día de la semana:
            mean   std  count
DayName                      
Monday     41.13  7.51     52
Tuesday    43.75  6.79     52
Wednesday  43.85  9.06     52
Thursday   43.08  6.40     53
Friday     41.96  6.28     52
Saturday   41.19  7.88     52
Sunday     38.88  6.19     52

Nacimientos por mes:
            mean   std   sum
MonthName                   
January    39.13  7.74  1213
February   41.00  7.94  1148
March      39.29  6.90  1218
April      39.83  7.21  1195
May        38.97  6.14  1208
June 

In [46]:
# Crear figura principal con diseño de dashboard
fig = plt.figure(figsize=(20, 12))
fig.patch.set_facecolor('#f8f9fa')

# Título principal del dashboard
fig.suptitle('DASHBOARD: ANÁLISIS DE NACIMIENTOS FEMENINOS DIARIOS - 1959', 
             fontsize=24, fontweight='bold', y=0.98)

# Crear una cuadrícula personalizada
gs = GridSpec(3, 3, figure=fig, height_ratios=[1, 1.2, 0.8], width_ratios=[1, 1, 1],
              hspace=0.3, wspace=0.25)

# ===========================
# 1. BOXPLOT POR DÍA DE LA SEMANA (Superior izquierda)
# ===========================
ax1 = fig.add_subplot(gs[0, :2])
ax1.set_facecolor('#ffffff')

# Preparar datos para boxplot
df_plot = df.copy()
df_plot['DayName'] = pd.Categorical(df_plot['DayName'], categories=day_order, ordered=True)

# Crear boxplot mejorado
bp = df_plot.boxplot(column='Births', by='DayName', ax=ax1, patch_artist=True,
                     boxprops=dict(facecolor='lightblue', alpha=0.8),
                     medianprops=dict(color='darkred', linewidth=2),
                     whiskerprops=dict(color='gray', linewidth=1.5),
                     capprops=dict(color='gray', linewidth=1.5),
                     flierprops=dict(marker='o', markerfacecolor='red', markersize=6, alpha=0.6))

# Personalizar
ax1.set_title('Distribución de Nacimientos por Día de la Semana', 
              fontsize=16, fontweight='bold', pad=15)
ax1.set_xlabel('Día de la Semana', fontsize=12, fontweight='bold')
ax1.set_ylabel('Número de Nacimientos', fontsize=12, fontweight='bold')
ax1.set_xticklabels(['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom'], rotation=0)
ax1.grid(True, alpha=0.3, axis='y')

# Añadir línea de media general
ax1.axhline(y=df['Births'].mean(), color='green', linestyle='--', 
            linewidth=2, alpha=0.7, label=f'Media general: {df["Births"].mean():.1f}')
ax1.legend(loc='upper right')

# Eliminar título automático del boxplot
fig.texts = [text for text in fig.texts if 'Boxplot grouped by' not in text.get_text()]

# ===========================
# 2. COMPARACIÓN FIN DE SEMANA VS DÍAS LABORABLES (Superior derecha)
# ===========================
ax2 = fig.add_subplot(gs[0, 2])
ax2.set_facecolor('#ffffff')

# Calcular comparación
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6])
weekend_comparison = df.groupby('IsWeekend')['Births'].agg(['mean', 'std', 'count'])
weekend_comparison.index = ['Días laborables', 'Fin de semana']

# Crear gráfico de barras con barras de error
bars = ax2.bar(weekend_comparison.index, weekend_comparison['mean'], 
                yerr=weekend_comparison['std'], capsize=10,
                color=['#3498db', '#e74c3c'], alpha=0.8, edgecolor='black', linewidth=1.5)

# Añadir valores encima de las barras
for i, (idx, row) in enumerate(weekend_comparison.iterrows()):
    ax2.text(i, row['mean'] + row['std'] + 0.5, f"{row['mean']:.1f}", 
             ha='center', va='bottom', fontsize=14, fontweight='bold')
    # Añadir diferencia porcentual
    if i == 1:
        diff_pct = ((weekend_comparison.iloc[0]['mean'] - row['mean']) / row['mean'] * 100)
        ax2.text(0.5, row['mean'] + 5, f'+{diff_pct:.1f}%', 
                 ha='center', fontsize=12, color='green', fontweight='bold')

ax2.set_title('Promedio de Nacimientos:\nDías Laborables vs Fin de Semana', 
              fontsize=14, fontweight='bold', pad=15)
ax2.set_ylabel('Promedio de Nacimientos', fontsize=12, fontweight='bold')
ax2.set_ylim(0, weekend_comparison['mean'].max() * 1.3)
ax2.grid(True, alpha=0.3, axis='y')

# ===========================
# 3. SERIE TEMPORAL CON MEDIAS MÓVILES (Centro)
# ===========================
ax3 = fig.add_subplot(gs[1, :])
ax3.set_facecolor('#ffffff')

# Calcular medias móviles
df['MA7'] = df['Births'].rolling(window=7).mean()
df['MA30'] = df['Births'].rolling(window=30).mean()

# Graficar serie temporal
df['Births'].plot(ax=ax3, alpha=0.4, label='Datos diarios', color='lightgray', linewidth=0.8)
df['MA7'].plot(ax=ax3, label='Media móvil 7 días', linewidth=2.5, color='#2ecc71')
df['MA30'].plot(ax=ax3, label='Media móvil 30 días', linewidth=2.5, color='#e74c3c')

# Resaltar máximo y mínimo
max_idx = df['Births'].idxmax()
min_idx = df['Births'].idxmin()
ax3.scatter(max_idx, df.loc[max_idx, 'Births'], color='red', s=150, zorder=5, 
           label=f'Máximo: {df.loc[max_idx, "Births"]}')
ax3.scatter(min_idx, df.loc[min_idx, 'Births'], color='darkred', s=150, zorder=5,
           label=f'Mínimo: {df.loc[min_idx, "Births"]}')

# Añadir anotaciones
ax3.annotate(f'{df.loc[max_idx, "Births"]} nacimientos\n{max_idx.strftime("%d %b")}', 
             xy=(max_idx, df.loc[max_idx, 'Births']), 
             xytext=(10, 20), textcoords='offset points',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='yellow', alpha=0.7),
             arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

ax3.set_title('Serie Temporal de Nacimientos con Medias Móviles', 
              fontsize=18, fontweight='bold', pad=15)
ax3.set_xlabel('Fecha', fontsize=12, fontweight='bold')
ax3.set_ylabel('Número de Nacimientos', fontsize=12, fontweight='bold')
ax3.legend(loc='upper left', fontsize=11, framealpha=0.9)
ax3.grid(True, alpha=0.3)

# Sombrear fines de semana
for i in range(len(df)):
    if df.index[i].dayofweek >= 5:  # Sábado o domingo
        ax3.axvspan(df.index[i], df.index[i] + pd.Timedelta(days=1), 
                   alpha=0.1, color='orange')

# ===========================
# 4. TOTAL DE NACIMIENTOS POR MES (Inferior)
# ===========================
ax4 = fig.add_subplot(gs[2, :])
ax4.set_facecolor('#ffffff')

# Calcular totales mensuales
monthly_births = df.groupby('Month')['Births'].agg(['sum', 'mean', 'std'])
month_names = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 
               'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

# Crear gráfico de barras
bars = ax4.bar(range(1, 13), monthly_births['sum'], color='coral', alpha=0.8, 
                edgecolor='black', linewidth=1.5)

# Colorear el mes con más y menos nacimientos
max_month = monthly_births['sum'].idxmax()
min_month = monthly_births['sum'].idxmin()
bars[max_month-1].set_color('#27ae60')
bars[min_month-1].set_color('#c0392b')

# Añadir línea de promedio
ax4.axhline(y=monthly_births['sum'].mean(), color='blue', linestyle='--', 
            linewidth=2, alpha=0.7, label=f'Promedio mensual: {monthly_births["sum"].mean():.0f}')

# Añadir valores encima de las barras
for i, (idx, row) in enumerate(monthly_births.iterrows()):
    ax4.text(i+1, row['sum'] + 10, f"{row['sum']:,}", 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

ax4.set_title('Total de Nacimientos por Mes', fontsize=16, fontweight='bold', pad=15)
ax4.set_xlabel('Mes', fontsize=12, fontweight='bold')
ax4.set_ylabel('Total de Nacimientos', fontsize=12, fontweight='bold')
ax4.set_xticks(range(1, 13))
ax4.set_xticklabels(month_names, rotation=0)
ax4.legend(loc='upper right')
ax4.grid(True, alpha=0.3, axis='y')

# ===========================
# PANEL DE ESTADÍSTICAS RESUMEN
# ===========================
# Crear un texto con estadísticas clave
stats_text = f"""
ESTADÍSTICAS CLAVE:
• Total de nacimientos en 1959: {df['Births'].sum():,}
• Promedio diario: {df['Births'].mean():.1f} ± {df['Births'].std():.1f}
• Diferencia laborables vs fin de semana: {weekend_comparison.iloc[0]['mean'] - weekend_comparison.iloc[1]['mean']:.1f} nacimientos/día
• Mes con más nacimientos: {month_names[max_month-1]} ({monthly_births.loc[max_month, 'sum']:,})
• Mes con menos nacimientos: {month_names[min_month-1]} ({monthly_births.loc[min_month, 'sum']:,})
"""

# Añadir el texto al dashboard
fig.text(0.02, 0.02, stats_text, fontsize=12, 
         bbox=dict(boxstyle="round,pad=0.5", facecolor='lightgray', alpha=0.8),
         verticalalignment='bottom')

# Ajustar el layout
plt.tight_layout()

# Guardar el dashboard
plt.savefig('dashboard_nacimientos.png', dpi=300, bbox_inches='tight', facecolor='#f8f9fa')
plt.show()

# Imprimir resumen adicional
print("=" * 60)
print("DASHBOARD DE NACIMIENTOS GENERADO EXITOSAMENTE")
print("=" * 60)
print("\nInsights principales visualizados:")
print("1. Clara diferencia entre días laborables y fines de semana")
print("2. Patrón semanal consistente visible en las medias móviles")
print("3. Variación mensual relativamente estable")
print("4. Los domingos muestran la menor cantidad de nacimientos")
print("\nArchivo guardado como: dashboard_nacimientos.png")
print("=" * 60)

DASHBOARD DE NACIMIENTOS GENERADO EXITOSAMENTE

Insights principales visualizados:
1. Clara diferencia entre días laborables y fines de semana
2. Patrón semanal consistente visible en las medias móviles
3. Variación mensual relativamente estable
4. Los domingos muestran la menor cantidad de nacimientos

Archivo guardado como: dashboard_nacimientos.png


# 2. Promedios Móviles:

# 3. Alisamiento Exponencial:

# 4. HOLT-WINTERS

# 5. SARIMA:

# 6. Prophet:

# 7. Comparación y Evaluación:
